# NB06 — EfficientNet-B4 on DF40: Completing the Architecture x Dataset Grid

**Why:** the architecture-generality claim (paper 5.4) currently rests on EfficientNet-B4's
4 FF++ points only. Running EffNet on DF40 gives it points on BOTH datasets — the full
architecture x dataset grid (Xception: FF++ + DF40; EffNet: FF++ + DF40). Makes the claim
symmetric and much stronger.

**Flow (reuses NB05/NB03 exactly):** load EffNet FS checkpoint -> score 8 DF40 generators ->
calibrate -> add points -> recompute combined coupling with the new EffNet-DF40 cells.

EffNet preprocessing = same as Xception (256, mean/std 0.5) — confirmed in NB05.


## Cell 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, glob, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
DFB = f"{REPO}/external/DeepfakeBench"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"{PARENT}/{f}"): subprocess.run(f'cp "{PARENT}/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
print("installing deps..."); subprocess.run("pip -q install efficientnet_pytorch timm einops kornia simplejson", shell=True)
print("EffNet ckpt:", os.path.exists(f"{REPO}/weights/train_on_fs/efficientnetb4.pth"))
print("DF40 core zips:", len(glob.glob(f"{REPO}/data/df40_core/test/*.zip")))

Mounted at /content/drive
installing deps...
EffNet ckpt: True
DF40 core zips: 9


## Cell 2 — Load EfficientNet-B4 (FS) detector

In [3]:
import os, glob
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
ND = f"{REPO}/external/DeepfakeBench/training/networks"
print("=== networks/ contents ===")
for f in sorted(glob.glob(f"{ND}/*.py")):
    print("  ", os.path.basename(f))
# what does networks/__init__.py import?
init = f"{ND}/__init__.py"
if os.path.exists(init):
    print("\n=== networks/__init__.py ===")
    print(open(init).read()[:1500])

=== networks/ contents ===
   __init__.py
   adaface.py
   base_backbone.py
   cls_hrnet.py
   efficientnetb4.py
   iresnet.py
   iresnet_iid.py
   mesonet.py
   resnet.py
   resnet34.py
   time_transformer.py
   vgg.py
   xception.py
   xception_ffd.py
   xception_sladd.py

=== networks/__init__.py ===
import os
import sys
current_file_path = os.path.abspath(__file__)
parent_dir = os.path.dirname(os.path.dirname(current_file_path))
project_root_dir = os.path.dirname(parent_dir)
sys.path.append(parent_dir)
sys.path.append(project_root_dir)

from metrics.registry import BACKBONE

from .xception import Xception
from .mesonet import Meso4, MesoInception4
from .resnet34 import ResNet34
from .efficientnetb4 import EfficientNetB4
from .xception_sladd import Xception_SLADD



In [4]:
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
path = f"{REPO}/src/inference.py"
src = open(path).read()

# the current fragile block hardcodes networks/xception.py; replace with package import
old = '''    import importlib as _il
    try:
        _il.import_module("networks")
    except Exception:
        # networks/__init__ may also pull heavy deps; import the xception net directly
        _bspec = importlib.util.spec_from_file_location(
            "networks.xception", f"{train_dir}/networks/xception.py")
        if _bspec is not None:
            _bmod = importlib.util.module_from_spec(_bspec)
            sys.modules.setdefault("networks", types.ModuleType("networks"))
            sys.modules["networks"].__path__ = [f"{train_dir}/networks"]
            sys.modules["networks.xception"] = _bmod
            _bspec.loader.exec_module(_bmod)'''

new = '''    # import the networks PACKAGE so ALL backbones register (its __init__ imports
    # xception, efficientnetb4, resnet34, mesonet, etc. -> populates BACKBONE registry).
    # Must run with training/ on sys.path (it is, set above).
    import importlib as _il
    if "networks" in sys.modules:
        del sys.modules["networks"]   # force fresh registration on this runtime
    try:
        _il.import_module("networks")
    except Exception as _e:
        print(f"  [inference] networks package import warning: {_e}")
        # fallback: import the specific backbone module by file
        _bb_file = {"xception":"xception.py","efficientnetb4":"efficientnetb4.py",
                    "f3net":"xception.py","clip":"clip.py"}.get(backbone_name, "xception.py")
        _bspec = importlib.util.spec_from_file_location(
            f"networks.{backbone_name}", f"{train_dir}/networks/{_bb_file}")
        if _bspec is not None:
            sys.modules.setdefault("networks", types.ModuleType("networks"))
            sys.modules["networks"].__path__ = [f"{train_dir}/networks"]
            _bmod = importlib.util.module_from_spec(_bspec)
            sys.modules[f"networks.{backbone_name}"] = _bmod
            _bspec.loader.exec_module(_bmod)'''

src = src.replace(old, new)
open(path, "w").write(src)
print("patched: networks imported as package (registers all backbones incl. EffNet)")

patched: networks imported as package (registers all backbones incl. EffNet)


In [5]:
import sys, importlib.util, torch, hashlib
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"
for k in list(sys.modules.keys()):
    if k.startswith("detectors") or k.startswith("networks") or k=="metrics" or k.startswith("metrics.") or k=="inference":
        del sys.modules[k]
sys.path = [p for p in sys.path if p not in (f"{DFB}/training", f"{REPO}/src", DFB)]
sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")
spec = importlib.util.spec_from_file_location("inference", f"{REPO}/src/inference.py")
inference = importlib.util.module_from_spec(spec); sys.modules["inference"]=inference
spec.loader.exec_module(inference)

ckpt = f"{REPO}/weights/train_on_fs/efficientnetb4.pth"
model, device, info = inference.load_detector(dfb_root=DFB, backbone_name="efficientnetb4", ckpt_path=ckpt)
print("EffNet load:", info, "device:", device)

EffNet load: {'missing': 0, 'unexpected': 0} device: cuda


## Cell 3 — Score 8 DF40 generators with EffNet (two-pass: score now, calibrate next)

In [6]:
import os, glob, json, zipfile, shutil
import pandas as pd, importlib.util, sys
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

spec = importlib.util.spec_from_file_location("data_prep", f"{REPO}/src/data_prep.py")
data_prep = importlib.util.module_from_spec(spec); sys.modules["data_prep"]=data_prep
spec.loader.exec_module(data_prep)

CDF_REAL = f"{REPO}/data/frames/Celeb-DF-v2"
real_index = {"/".join(fp.split("/")[-2:]): fp for fp in glob.glob(f"{CDF_REAL}/**/frames/**/*.png", recursive=True)}
print(f"real index: {len(real_index)}\n")

methods = ["simswap","inswap","blendface","fomm","facevid2vid","StyleGAN2","sd2.1","ddim"]
for method in methods:
    print(f"=== {method} (EffNet) ===")
    zp = f"{REPO}/data/df40_core/test/{method}.zip"
    jpath = f"{REPO}/data/df40/dataset_json/{method}_cdf.json"
    if not (os.path.exists(zp) and os.path.exists(jpath)):
        print("  missing, skip"); continue
    fdir = f"/content/df40eff_{method}"
    if not os.path.isdir(fdir):
        os.makedirs(fdir, exist_ok=True)
        with zipfile.ZipFile(zp) as z: z.extractall(fdir)
    fake_index = {"/".join(fp.split("/")[-2:]): fp for fp in glob.glob(f"{fdir}/**/*.png", recursive=True)}
    df = data_prep.build_manifest_from_json(f"{method}_cdf", jpath, frames_root=None)
    def remap(row):
        key = "/".join(row["frame_path"].split("/")[-2:])
        return fake_index.get(key) if row["label"]==1 else real_index.get(key)
    df["frame_path"] = df.apply(remap, axis=1)
    df = df[df["frame_path"].notna()].reset_index(drop=True)
    if df['label'].nunique() < 2:
        print("  one label, skip"); shutil.rmtree(fdir, ignore_errors=True); continue
    scores = inference.score_manifest(model, device, df, batch_size=64, verbose=False)
    scores.to_parquet(f"{REPO}/reports/scores/effnetb4_df40_{method}.parquet", index=False)
    auc = roc_auc_score(scores.label, scores.prob_fake)
    print(f"  AUC = {auc:.4f}  (n={len(scores)})")
    shutil.rmtree(fdir, ignore_errors=True)
print("\nPass done — EffNet DF40 scores saved.")

real index: 16420

=== simswap (EffNet) ===
  AUC = 0.8408  (n=25942)
=== inswap (EffNet) ===
  AUC = 0.7434  (n=19053)
=== blendface (EffNet) ===
  AUC = 0.9284  (n=25943)
=== fomm (EffNet) ===
  AUC = 0.7180  (n=25898)
=== facevid2vid (EffNet) ===
  AUC = 0.6661  (n=25510)
=== StyleGAN2 (EffNet) ===
  AUC = 0.4850  (n=33794)
=== sd2.1 (EffNet) ===
  AUC = 0.7928  (n=33794)
=== ddim (EffNet) ===
  AUC = 0.7640  (n=33794)

Pass done — EffNet DF40 scores saved.


## Cell 4 — Calibrate EffNet-DF40 + the full architecture x dataset coupling

In [7]:
import sys, importlib.util, os, glob
import numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k=="calibration": del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

fam = {"simswap":"FS","inswap":"FS","blendface":"FS","fomm":"FR","facevid2vid":"FR",
       "StyleGAN2":"EFS","sd2.1":"EFS","ddim":"EFS"}
rows = []
for f in sorted(glob.glob(f"{REPO}/reports/scores/effnetb4_df40_*.parquet")):
    method = os.path.basename(f).replace("effnetb4_df40_","").replace(".parquet","")
    s = pd.read_parquet(f)
    p = s.prob_fake.values.astype(float); y = s.label.values.astype(int)
    g = s.identity_id.values if "identity_id" in s.columns else None
    ci, ti, _ = cal.leakage_safe_split(y, groups=g, calib_frac=0.5, seed=42)
    p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
    ece_raw = met.ece(p[ti], y[ti], n_bins=15, scheme="equal_mass")
    ece_cal = met.ece(p_cal, y[ti], n_bins=15, scheme="equal_mass")
    auc = met.roc_auc(p[ti], y[ti]) if hasattr(met,"roc_auc") else float("nan")
    rows.append({"method":f"effDF40_{method}","architecture":"EfficientNet-B4","source":"DF40",
                 "family":fam.get(method,"?"),"AUC":auc,"ECE_raw":ece_raw,"ECE_cal":ece_cal})
    print(f"  {method:14s} [{fam.get(method,'?'):3s}] AUC={auc:.4f}  ECE_cal={ece_cal:.4f}")
eff_df40 = pd.DataFrame(rows)
eff_df40.to_csv(f"{REPO}/reports/calibration/coupling_effnetb4_df40.csv", index=False)

# EffNet-only coupling now spans BOTH datasets
eff_ffpp = pd.read_csv(f"{REPO}/reports/calibration/coupling_effnetb4_ffpp.csv")[["AUC","ECE_cal"]]
eff_all = pd.concat([eff_ffpp, eff_df40[["AUC","ECE_cal"]]])
r,p = pearsonr(eff_all.AUC, eff_all.ECE_cal)
print(f"\nEfficientNet-B4 coupling, BOTH datasets ({len(eff_all)} pts): r={r:.3f} (p={p:.4f})")

# grand combined: everything
prev = pd.read_csv(f"{REPO}/reports/calibration/coupling_all_16pts.csv")[["AUC","ECE_cal"]]
frm = pd.read_csv(f"{REPO}/reports/calibration/coupling_df40_FRmismatch.csv")[["AUC","ECE_cal"]]
grand = pd.concat([prev, frm, eff_ffpp, eff_df40[["AUC","ECE_cal"]]])
rg,pg = pearsonr(grand.AUC, grand.ECE_cal); rsg,psg = spearmanr(grand.AUC, grand.ECE_cal)
print(f"GRAND combined ({len(grand)} pts): Pearson r={rg:.3f} (p={pg:.2e}) | Spearman={rsg:.3f}")

  StyleGAN2      [EFS] AUC=0.4658  ECE_cal=0.1808
  blendface      [FS ] AUC=0.9433  ECE_cal=0.0368
  ddim           [EFS] AUC=0.7450  ECE_cal=0.0318
  facevid2vid    [FR ] AUC=0.7113  ECE_cal=0.1983
  fomm           [FR ] AUC=0.7668  ECE_cal=0.1154
  inswap         [FS ] AUC=0.7214  ECE_cal=0.0992
  sd2.1          [EFS] AUC=0.7767  ECE_cal=0.0334
  simswap        [FS ] AUC=0.8607  ECE_cal=0.0449

EfficientNet-B4 coupling, BOTH datasets (12 pts): r=-0.769 (p=0.0035)
GRAND combined (32 pts): Pearson r=-0.807 (p=2.47e-08) | Spearman=-0.818


## Cell 5 — Updated headline figure (full grid) + commit

In [ ]:
import pandas as pd, numpy as np, os, glob
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# assemble all points with architecture + dataset tags
prev16 = pd.read_csv(f"{REPO}/reports/calibration/coupling_all_16pts.csv")
prev16["architecture"]="Xception"
frm = pd.read_csv(f"{REPO}/reports/calibration/coupling_df40_FRmismatch.csv"); frm["architecture"]="Xception"; frm["source"]="DF40"
efff = pd.read_csv(f"{REPO}/reports/calibration/coupling_effnetb4_ffpp.csv"); efff["architecture"]="EfficientNet-B4"; efff["source"]="FF++"
effd = pd.read_csv(f"{REPO}/reports/calibration/coupling_effnetb4_df40.csv")
allp = pd.concat([prev16[["AUC","ECE_cal","architecture","source"]],
                  frm[["AUC","ECE_cal","architecture","source"]],
                  efff[["AUC","ECE_cal","architecture","source"]],
                  effd[["AUC","ECE_cal","architecture","source"]]]).reset_index(drop=True)

r,p = pearsonr(allp.AUC, allp.ECE_cal)
fig, ax = plt.subplots(figsize=(8,5.5))
# marker by architecture, color by dataset
markers = {"Xception":"o","EfficientNet-B4":"^"}
colors = {"FF++":"#C0392B","DF40":"#2471A3"}
for (arch,src), g in allp.groupby(["architecture","source"]):
    ax.scatter(g.AUC, g.ECE_cal, s=85, alpha=0.8, marker=markers.get(arch,"o"),
               color=colors.get(src,"gray"), edgecolor="white", linewidth=1.1,
               label=f"{arch} / {src} (n={len(g)})", zorder=3)
x,y=allp.AUC.values, allp.ECE_cal.values
b,a=np.polyfit(x,y,1); xs=np.linspace(x.min()-.02,x.max()+.02,100)
ax.plot(xs,a+b*xs,"k--",lw=1.8,label="OLS (all)",zorder=2)
ax.annotate(f"r={r:.2f} (p={p:.1e}), n={len(allp)}",xy=(.97,.95),xycoords="axes fraction",
            ha="right",va="top",fontsize=11,bbox=dict(boxstyle="round,pad=0.4",fc="white",ec="gray"))
ax.set_xlabel("Detection competence (AUC)",fontsize=12); ax.set_ylabel("ECE$_{cal}$",fontsize=12)
ax.set_title("Competence-Calibration Coupling: full architecture x dataset grid",fontsize=12)
ax.legend(loc="upper right",fontsize=8.5); ax.grid(True,alpha=0.25); plt.tight_layout()
out=f"{REPO}/figures/coupling_full_grid.png"
plt.savefig(out,dpi=300,bbox_inches="tight"); plt.show()
allp.to_csv(f"{REPO}/reports/calibration/coupling_full_grid.csv", index=False)
print(f"saved figure + {len(allp)}-point grid CSV")

## Cell 6 — Commit

In [ ]:
import os, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
os.chdir(REPO)
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"):
        subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run("git add reports/scores/effnetb4_df40_*.parquet reports/calibration/coupling_effnetb4_df40.csv reports/calibration/coupling_full_grid.csv figures/coupling_full_grid.png notebooks/NB06_effnet_df40.ipynb", shell=True)
r = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print(r.stdout)
print(">>> review, then commit in a follow-up with the grand-combined r")